# Preparación histórica de los datos
Código del notebook académico original, reorganizado sin consignas ni comentarios del revisor. No ejecutado en esta revisión: faltan `tomtom_traffic.csv` y `oecd_city_economy.csv`.

Coloca esos archivos en `data/raw/` para revisar el proceso. Los paths se adaptaron al repositorio; las transformaciones se conservan. Este notebook genera `data/ladb_mobility_economy_2024_reconstruido.csv` para no sobrescribir el CSV aportado.

Antes de ejecutarlo, revisar formatos numéricos, claves ciudad-país-año, duplicados y cobertura de la unión. La unión histórica usa solo ciudad y año y no valida cardinalidad; no debe suponerse que sea segura para otras fuentes. La eliminación de puntos del PIB presupone un formato de miles específico.


In [ ]:
from pathlib import Path
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
traffic = pd.read_csv(ROOT / 'data/raw/tomtom_traffic.csv')
eco = pd.read_csv(ROOT / 'data/raw/oecd_city_economy.csv')


In [ ]:
traffic = traffic.rename(columns={'Country': 'country','UpdateTimeUTC': 'update_time_utc'})
traffic.columns


In [ ]:
eco = eco.rename(columns={'Country': 'country',})
eco.columns


In [ ]:
traffic['update_time_utc'] = pd.to_datetime(traffic['update_time_utc'], errors='coerce')
traffic['UpdateTimeUTCWeekAgo'] = pd.to_datetime(traffic['UpdateTimeUTCWeekAgo'], errors='coerce')

traffic.info()


In [ ]:
print(eco.columns)
eco = eco.rename(columns={
    'City GDP/capita': 'city_gdp_capita',
    'Unemployment %': 'unemployment_pct',
    'Population (M)': 'population_m'
})
eco['city_gdp_capita'] = (
    eco['city_gdp_capita']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)
eco['unemployment_pct'] = (
eco['unemployment_pct'] 
.astype(str)
.str.replace('%', '', regex=False)
.str.replace(",", ".", regex=False)
.astype(float)
)
eco['population_m'] = (eco['population_m'] 
.astype(str)
.str.replace(",", ".", regex=False)
.astype(float)
)
eco['population'] = eco['population_m'] * 1_000_000
eco.info()
eco.head(3)


In [ ]:
traffic['Year'] = traffic['update_time_utc'].dt.year


traffic.head(3)


In [ ]:
traffic_2024= traffic[traffic['Year'] == 2024].copy()
eco_2024 = eco[eco['Year'] == 2024].copy()
display(traffic_2024.head())
display(eco_2024.head())


In [ ]:
traffic_city_year_2024 = (
    traffic_2024
    .groupby(['City', 'country', 'Year'])
    .agg({
        'JamsDelay': 'mean',
        'TrafficIndexLive': 'mean',
        'JamsLengthInKms': 'mean',
        'JamsCount': 'mean',
        'MinsDelay': 'mean'
    })
    .reset_index()
)


traffic_city_year_2024.head()


In [ ]:
left_cols = [
    'City', 'country', 'Year',
    'JamsDelay', 'TrafficIndexLive',
    'JamsLengthInKms', 'JamsCount', 'MinsDelay'
]

right_cols = ['City','Year','city_gdp_capita','unemployment_pct','PM2.5 (μg/m³)','population']

traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

merged = pd.merge(
    traffic_2024_small,
    eco_2024_small,
    on=['City', 'Year'],
    how='inner'
)

merged.head()


In [ ]:
merged.to_csv(ROOT / 'data/ladb_mobility_economy_2024_reconstruido.csv', index=False)
